# DEMO TỪNG HÀM / METHOD LANGCHAIN PDF QA

Notebook dùng **chính các hàm trong ZIP mới**, PDF mẫu kèm theo. Chạy lần lượt `Shift+Enter`. Cần cài đặt bằng `01_CAI_DAT_LAN_DAU.bat`. **Kết quả cần được chạy thật trên máy bạn, không có dữ liệu benchmark bịa sẵn.**

In [1]:
from pathlib import Path
import os, sys, time
# Khi mở từ 03_MO_NOTEBOOK.bat, JupyterLab chạy ở thư mục dự án.
ROOT = Path.cwd()
if not (ROOT / 'config.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print('Thư mục dự án:', ROOT)

Thư mục dự án: <LOCAL_PROJECT_DIR>


## Method 1. `list_pdfs()` — Input: thư mục / Output: danh sách PDF

In [2]:
from config import PDF_DIR, INDEX_DIR, EMBEDDING_MODEL, LLM_MODEL
from src.pdf_reader import list_pdfs, load_pdfs
pdfs = list_pdfs(PDF_DIR)
print([p.name for p in pdfs])

['TAI_LIEU_DEMO_CAU_LAC_BO_SACH.pdf']


## Method 2. `load_pdfs()` và `PyPDFLoader.load()` — Input: các PDF / Output: Document theo trang

In [3]:
pages = load_pdfs(PDF_DIR)
print('Số trang có chữ:', len(pages))
print('Metadata:', pages[0].metadata)
print('Nội dung mẫu:', pages[0].page_content[:400])

Số trang có chữ: 3
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-11-03T06:03:38+00:00', 'author': 'anonymous', 'keywords': '', 'moddate': '2026-11-03T06:03:38+00:00', 'subject': 'unspecified', 'title': 'untitled', 'trapped': '/False', 'source': 'TAI_LIEU_DEMO_CAU_LAC_BO_SACH.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}
Nội dung mẫu: TÀI LIỆU THỰC HÀNH LANGCHAIN RAG
Bản minh họa (giả định)  •  Trang 1/3
GIỚI THIỆU CÂU LẠC BỘ SÁCH SAO MAI
Đây là tài liệu mẫu được tạo riêng để thực hành hỏi đáp PDF; thông tin và tổ chức đều là giả
định.
Câu lạc bộ Sách Sao Mai được thành lập năm 2024 nhằm khuyến khích thói quen đọc sách, chia
sẻ kiến thức và kỹ năng tự học.
Câu lạc bộ tổ chức buổi sinh hoạt định kỳ vào thứ Bảy hàng tuần, từ 08:0


## Verify the supplied PDF (ground truth before model evaluation)

The book-club sample PDF is fictional. We check known facts directly from the text so retrieval has an objective reference. This test does **not** run the LLM.


In [4]:
reference_facts = {
    'founded_in_2024': ('2024' in pages[0].page_content),
    'meets_on_saturday': ('thứ Bảy' in pages[0].page_content),
    'borrowing_14_days': ('14 ngày' in pages[1].page_content),
    'two_books': ('2 cuốn' in pages[1].page_content),
    'does_not_define_langchain': not any('LangChain là một công cụ' in p.page_content for p in pages),
}
for label, result in reference_facts.items(): print(label, ':', 'PASS' if result else 'CHECK SOURCE')
assert all(reference_facts.values()), 'Reference PDF was changed; update the verification statements.'


founded_in_2024 : PASS
meets_on_saturday : PASS
borrowing_14_days : PASS
two_books : PASS
does_not_define_langchain : PASS


## Method 3. `split_documents()` — Input: Document + tham số / Output: chunks

In [5]:
from src.text_chunker import split_documents
chunks = split_documents(pages, chunk_size=900, chunk_overlap=150)
print('Số chunks:', len(chunks))
print('Chunk đầu:', chunks[0].page_content[:400])

Số chunks:

 3
Chunk đầu: TÀI LIỆU THỰC HÀNH LANGCHAIN RAG
Bản minh họa (giả định)  •  Trang 1/3
GIỚI THIỆU CÂU LẠC BỘ SÁCH SAO MAI
Đây là tài liệu mẫu được tạo riêng để thực hành hỏi đáp PDF; thông tin và tổ chức đều là giả
định.
Câu lạc bộ Sách Sao Mai được thành lập năm 2024 nhằm khuyến khích thói quen đọc sách, chia
sẻ kiến thức và kỹ năng tự học.
Câu lạc bộ tổ chức buổi sinh hoạt định kỳ vào thứ Bảy hàng tuần, từ 08:0


## Method 4. `make_embeddings()` — Input: tên model / Output: vector hoá đa ngôn ngữ

In [6]:
from src.vector_store import make_embeddings
emb = make_embeddings(EMBEDDING_MODEL)
vec = emb.embed_query('Câu lạc bộ họp vào thứ mấy?')
print('Chiều vector:', len(vec), '5 giá trị đầu:', vec[:5])

<LOCAL_PROJECT_DIR>\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2991.19it/s]

Chiều vector: 384 5 giá trị đầu: [0.5625863075256348, -0.17693516612052917, -0.06159874051809311, 0.12889914214611053, -0.06111079454421997]


## Method 5. `build_vector_store()` — Input: chunks + embedding / Output: FAISS

In [7]:
from src.vector_store import build_vector_store, save_vector_store, load_vector_store
store = build_vector_store(chunks, emb)
print('Số vector:', store.index.ntotal)

Số vector: 3


## Method 6. `save_vector_store()` + `load_vector_store()` — Input: FAISS / Output: index lưu & nạp

In [8]:
save_vector_store(store, INDEX_DIR)
loaded_store = load_vector_store(INDEX_DIR, emb)
print('Số vector đã nạp lại:', loaded_store.index.ntotal)

Số vector đã nạp lại: 3


## Method 7. `retrieve_documents()` — Input: câu hỏi / Output: top-k Document

In [9]:
from src.qa import retrieve_documents, format_context
question = 'Câu lạc bộ họp vào thứ mấy, mấy giờ?'
retrieved = retrieve_documents(loaded_store, question, k=2)
for i, doc in enumerate(retrieved,1):
    print(i, doc.metadata.get('source'), 'trang', doc.metadata.get('page',0)+1, doc.page_content[:180])

1 TAI_LIEU_DEMO_CAU_LAC_BO_SACH.pdf trang 3 TÀI LIỆU THỰC HÀNH LANGCHAIN RAG
Bản minh họa (giả định)  •  Trang 3/3
HOẠT ĐỘNG VÀ LIÊN HỆ
Câu lạc bộ tổ chức hoạt động "Giới thiệu sách tháng" vào thứ Bảy đầu tiên của mỗi tháng.
2 TAI_LIEU_DEMO_CAU_LAC_BO_SACH.pdf trang 1 TÀI LIỆU THỰC HÀNH LANGCHAIN RAG
Bản minh họa (giả định)  •  Trang 1/3
GIỚI THIỆU CÂU LẠC BỘ SÁCH SAO MAI
Đây là tài liệu mẫu được tạo riêng để thực hành hỏi đáp PDF; thông tin và 


## Method 8. `format_context()` — Input: chunks / Output: chuỗi context

In [10]:
context = format_context(retrieved)
print(context[:1000])

[1] File: TAI_LIEU_DEMO_CAU_LAC_BO_SACH.pdf; Trang: 3
TÀI LIỆU THỰC HÀNH LANGCHAIN RAG
Bản minh họa (giả định)  •  Trang 3/3
HOẠT ĐỘNG VÀ LIÊN HỆ
Câu lạc bộ tổ chức hoạt động "Giới thiệu sách tháng" vào thứ Bảy đầu tiên của mỗi tháng.
Mỗi học kỳ có một buổi trao đổi kỹ năng đọc hiểu và ghi chú. Thành viên có thể đăng ký thuyết
trình trong buổi sinh hoạt.
Quy định ứng xử: tôn trọng người phát biểu, không làm ồn trong giờ đọc sách, bảo quản tài liệu
chung.
Các thông tin trong tài liệu mẫu chỉ nhằm kiểm thử hệ thống, không thay thế quy định thực tế
của bất kỳ tổ chức nào.
Tài liệu không nêu tên hiệu trưởng, học phí, học bổng hay lịch thi. Nếu được hỏi, hệ thống nên
báo thiếu thông tin.
Đây là file PDF mẫu để thử truy vấn. Không phải tài liệu chính thức.

[2] File: TAI_LIEU_DEMO_CAU_LAC_BO_SACH.pdf; Trang: 1
TÀI LIỆU THỰC HÀNH LANGCHAIN RAG
Bản minh họa (giả định)  •  Trang 1/3
GIỚI THIỆU CÂU LẠC BỘ SÁCH SAO MAI
Đây là tài liệu mẫu được tạo riêng để thực hành hỏi đáp PDF; thông tin và tổ c

## Method 9. `generate_answer()` — Input: câu hỏi + context / Output: câu trả lời. Cần Ollama đang chạy.

In [11]:
from src.qa import generate_answer, answer_question
from app import check_ollama
check_ollama(LLM_MODEL)
answer = generate_answer(question, retrieved, LLM_MODEL)
print(answer)

Câu lạc bộ họp vào thứ Bảy, từ 08:00 đến 10:00.


## Method 10. `answer_question()` — Input: store, question, model, k / Output: dict có answer, sources

In [12]:
t0=time.perf_counter()
result=answer_question(loaded_store, question, LLM_MODEL, k=4)
print('Đáp án:', result['answer'])
print('Nguồn:', result['sources'])
print('Thời gian:', round(time.perf_counter()-t0,2),'giây')

Đáp án: Câu lạc bộ họp vào thứ Bảy, từ 08:00 đến 10:00.
Nguồn: [{'file': 'TAI_LIEU_DEMO_CAU_LAC_BO_SACH.pdf', 'page': 3, 'chunk_id': 3}, {'file': 'TAI_LIEU_DEMO_CAU_LAC_BO_SACH.pdf', 'page': 1, 'chunk_id': 1}, {'file': 'TAI_LIEU_DEMO_CAU_LAC_BO_SACH.pdf', 'page': 2, 'chunk_id': 2}]
Thời gian: 2.35 giây


## Method 11. `prepare_store()` — Index tự động khi thay đổi PDF / cấu hình

In [13]:
from src.pipeline import pdf_signature, build_manifest, prepare_store
print('Dấu hiệu PDF:', pdf_signature(PDF_DIR))
print('Manifest:', build_manifest(PDF_DIR))
store2, created_new, num_pages, num_chunks = prepare_store(PDF_DIR, INDEX_DIR)
print('Tạo mới:',created_new,'trang:',num_pages,'chunks:',num_chunks)

Dấu hiệu PDF: [{'name': 'TAI_LIEU_DEMO_CAU_LAC_BO_SACH.pdf', 'size': 50567, 'mtime_ns': 1793771394512260600}]
Manifest: {'pdfs': [{'name': 'TAI_LIEU_DEMO_CAU_LAC_BO_SACH.pdf', 'size': 50567, 'mtime_ns': 1793771394512260600}], 'chunk_size': 900, 'chunk_overlap': 150, 'embedding_model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3710.98it/s]

[INFO] PDF khong doi - su dung index FAISS da luu.
Tạo mới: False trang: None chunks: None


## Method I/O inventory: 17 project-level functions

These methods are defined in the supplied Python modules. Full real source code, example calls, and I/O contracts are in `tai_lieu/05_METHOD_INPUT_OUTPUT_CODE_EN.pdf`.


In [14]:
import inspect
from src import pdf_reader, text_chunker, vector_store, qa, pipeline
import app
function_groups = {
    'src/pdf_reader.py': [pdf_reader.list_pdfs,pdf_reader.load_pdfs],
    'src/text_chunker.py': [text_chunker.split_documents],
    'src/vector_store.py': [vector_store.make_embeddings,vector_store.build_vector_store,vector_store.save_vector_store,vector_store.load_vector_store],
    'src/qa.py': [qa.retrieve_documents,qa.format_context,qa.generate_answer,qa.answer_question],
    'src/pipeline.py': [pipeline.pdf_signature,pipeline.build_manifest,pipeline.prepare_store],
    'app.py': [app.check_ollama,app.save_history,app.main],
}
assert sum(len(group) for group in function_groups.values()) == 17
for module, functions in function_groups.items():
    for func in functions:
        print(f'{module} | {func.__name__}{inspect.signature(func)}')


src/pdf_reader.py | list_pdfs(folder: pathlib.Path) -> list[pathlib.Path]
src/pdf_reader.py | load_pdfs(folder: pathlib.Path) -> list[langchain_core.documents.base.Document]
src/text_chunker.py | split_documents(documents: list[langchain_core.documents.base.Document], chunk_size: int = 900, chunk_overlap: int = 150) -> list[langchain_core.documents.base.Document]
src/vector_store.py | make_embeddings(model_name: str) -> langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings
src/vector_store.py | build_vector_store(documents: list[langchain_core.documents.base.Document], embeddings: langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings) -> langchain_community.vectorstores.faiss.FAISS
src/vector_store.py | save_vector_store(store: langchain_community.vectorstores.faiss.FAISS, folder: pathlib.Path) -> None
src/vector_store.py | load_vector_store(folder: pathlib.Path, embeddings: langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings) -> langchain_commu

## Five real experiments A–E (change chunk_size, overlap, top-k)

All five experiments use the same sample PDF and question. The expected answer is **14 days on page 2**. We measure separate indexing and retrieval times, and record whether page 2 is retrieved. This is **not a full accuracy or LLM benchmark**. Values are produced only after executing this notebook on the actual machine.


In [15]:
# Real measurements are produced ONLY when this cell runs on a configured machine.
# Each setting rebuilds a temporary vector store in memory, not the main saved index.
configurations = [
    {'name':'A','chunk_size':500,'overlap':50,'k':2},
    {'name':'B','chunk_size':700,'overlap':100,'k':3},
    {'name':'C','chunk_size':900,'overlap':150,'k':4},
    {'name':'D','chunk_size':1100,'overlap':200,'k':3},
    {'name':'E','chunk_size':1300,'overlap':250,'k':5},
]
question = 'Thời hạn mượn sách là bao nhiêu ngày?'
experiment_results = []
for cfg in configurations:
    start = time.perf_counter()
    candidate_chunks = split_documents(pages,cfg['chunk_size'],cfg['overlap'])
    candidate_store = build_vector_store(candidate_chunks,emb)
    indexing_s = time.perf_counter()-start
    start = time.perf_counter()
    hits = retrieve_documents(candidate_store,question,cfg['k'])
    retrieval_s = time.perf_counter()-start
    pages_hit = [int(d.metadata.get('page',0))+1 for d in hits]
    row = {**cfg,
           'num_chunks':len(candidate_chunks),
           'num_retrieved':len(hits),
           'index_seconds':round(indexing_s,4),
           'retrieval_seconds':round(retrieval_s,4),
           'retrieved_pages':';'.join(map(str,pages_hit)),
           'top1_page_2': int(bool(pages_hit and pages_hit[0]==2)),
           'has_page_2_in_top_k':int(2 in pages_hit),
           'question':question}
    experiment_results.append(row)
for row in experiment_results:
    print(row)
print('IMPORTANT: top1_page_2 is a reference-page check for THIS one question, not overall answer accuracy.')


{'name': 'A', 'chunk_size': 500, 'overlap': 50, 'k': 2, 'num_chunks': 6, 'num_retrieved': 2, 'index_seconds': 0.1621, 'retrieval_seconds': 0.0177, 'retrieved_pages': '2;2', 'top1_page_2': 1, 'has_page_2_in_top_k': 1, 'question': 'Thời hạn mượn sách là bao nhiêu ngày?'}
{'name': 'B', 'chunk_size': 700, 'overlap': 100, 'k': 3, 'num_chunks': 5, 'num_retrieved': 3, 'index_seconds': 0.1518, 'retrieval_seconds': 0.0177, 'retrieved_pages': '2;3;1', 'top1_page_2': 1, 'has_page_2_in_top_k': 1, 'question': 'Thời hạn mượn sách là bao nhiêu ngày?'}
{'name': 'C', 'chunk_size': 900, 'overlap': 150, 'k': 4, 'num_chunks': 3, 'num_retrieved': 3, 'index_seconds': 0.0918, 'retrieval_seconds': 0.0166, 'retrieved_pages': '2;3;1', 'top1_page_2': 1, 'has_page_2_in_top_k': 1, 'question': 'Thời hạn mượn sách là bao nhiêu ngày?'}
{'name': 'D', 'chunk_size': 1100, 'overlap': 200, 'k': 3, 'num_chunks': 3, 'num_retrieved': 3, 'index_seconds': 0.0831, 'retrieval_seconds': 0.0155, 'retrieved_pages': '2;3;1', 'top1_p

## Export actual experiment data

The CSV is written **only after the previous cell finishes**. Values must not be copied or invented in advance.


In [16]:
import csv
OUT=ROOT/'ket_qua'
OUT.mkdir(parents=True,exist_ok=True)
with (OUT/'ket_qua_5_cau_hinh.csv').open('w',newline='',encoding='utf-8-sig') as f:
    writer=csv.DictWriter(f,fieldnames=experiment_results[0].keys())
    writer.writeheader();writer.writerows(experiment_results)
print('Đã lưu:', OUT/'ket_qua_5_cau_hinh.csv')

Đã lưu: <LOCAL_PROJECT_DIR>\ket_qua\ket_qua_5_cau_hinh.csv


## Kiểm tra tình huống trả lời ngoài PDF

Ví dụ: “Ai là hiệu trưởng?” PDF mẫu không nêu. LLM cần từ chối khẳng định không có nguồn.

In [17]:
outside = answer_question(loaded_store, 'Ai là hiệu trưởng trong tài liệu này?', LLM_MODEL, k=4)
print(outside['answer'])
print('Nguồn được truy xuất:', outside['sources'])

Tôi không tìm thấy thông tin này trong PDF.
Nguồn được truy xuất: [{'file': 'TAI_LIEU_DEMO_CAU_LAC_BO_SACH.pdf', 'page': 1, 'chunk_id': 1}, {'file': 'TAI_LIEU_DEMO_CAU_LAC_BO_SACH.pdf', 'page': 3, 'chunk_id': 3}, {'file': 'TAI_LIEU_DEMO_CAU_LAC_BO_SACH.pdf', 'page': 2, 'chunk_id': 2}]
